In [1]:
import numpy as np
import pandas as pd
import xtools as xt
import matplotlib.pyplot as plt
from pyqubo import Array, Constraint, solve_qubo
from openjij import SQASampler

from atm.flight.generator import ScenarioGenerator
from atm.optimizer.objective import get_obj_single_runway
from atm.separation import recat, faa

In [2]:
cf = xt.Config(dict(
    num=4,
    scenario=dict(
        interval=60,
        window=200,
        mode="mix",
        standard="recat",
        seed=0,
    ),
    penalty_coef=10,
))
sep = recat.TBS if cf.scenario.standard == "recat" else faa.TBS

In [3]:
def generate_scenario_and_obj(config, seed=None):
    if seed is not None:
        if hasattr(config, "config"):
            config.seed = seed
        else:
            config.set("seed", seed)
    gen = ScenarioGenerator(config.scenario)
    vols = gen(config.num)
    obj = get_obj_single_runway(vols, sep, penalty_coef=config.penalty_coef)
    return vols, obj

In [4]:
vols, obj = generate_scenario_and_obj(cf)
vols.to_dataframe()

,code,ready,due,category,operation
0,VOL0001,35,235,E,D
1,VOL0002,55,255,A,D
2,VOL0003,85,285,A,D
3,VOL0004,121,321,E,A


In [5]:
xc = np.arange(4)
ec = obj(xc)

In [6]:
nx = cf.num // 2
ks = [[vols[i], vols[i+1]] for i in range(nx)]

In [8]:
# Generate variable
xs = Array.create("x", shape=nx, vartype="BINARY")

In [9]:
def S(vol1, vol2):
    return sep(vol1, vol2)

In [10]:
# Intra-pair separation cost
e_in = Constraint(sum([
    S(vol1, vol2) * (1 - x) + S(vol2, vol1) for (vol1, vol2), x in zip(ks, xs)
]), "inner")
e_in


Constraint((((77.000000 * (1.000000 + (-1.000000 * Binary('x[1]')))) + 77.000000) + 0.000000 + (82.000000 * (1.000000 + (-1.000000 * Binary('x[0]')))) + 154.000000), 'inner')

In [11]:
# External-pair separation cost
lxs = list(xs)
e_ex = Constraint(sum([
    (S(j2, k2) - S(j2, k1)) * xk + (S(j1, k1) - S(j2, k1)) * xk + (S(j1, k2) - S(j2, k1)) * xj * xk
    for (j1, j2), (k1, k2), xj, xk in zip(ks[:-1], ks[1:], lxs[:-1], lxs[1:])
]), "external")
e_ex

Constraint((0.000000 + ((5.000000 * Binary('x[0]')) * Binary('x[1]')) + (0.000000 * Binary('x[1]')) + (5.000000 * Binary('x[1]'))), 'external')

In [12]:
e = e_in + e_ex

In [13]:
model = e.compile()
qubo, bias = model.to_qubo()

In [14]:
label_root = "x[{}]"

def to_matrix(model):
    return np.asarray([
        [
            qubo.get(
                (label_root.format(i1), label_root.format(i2)),
                0  # default value
            )
            for i2 in range(nx)
        ]
        for i1 in range(nx)
    ])

In [15]:
sampler = SQASampler(num_reads=10)
sampleset = sampler.sample_qubo(qubo)
sol = sampleset.first
dict_xp = sol.sample
cost_sample_xp = sol.energy + bias

In [16]:
xp = np.array([dict_xp.get(label_root.format(i)) for i in range(nx)])
mat = to_matrix(model)
xc = np.zeros_like(xp).reshape((nx, 1))
cost_sample_xc = bias
cost_sample_xc, cost_sample_xp

(390.0, 241.0)

In [17]:
T = nx * 120
np.exp(-cost_sample_xc / T), np.exp(-cost_sample_xp / T)

(0.19691167520419406, 0.3663497991338295)

In [18]:
sample_prob_ratio = np.exp(-cost_sample_xc / T) / np.exp(-cost_sample_xp / T)
sample_prob_ratio

0.5374963372977344

In [19]:
class MetropolisHastingsOptimizer:

    def __init__(self, num_qubit: int, num_reads:int = 10, seed: int | float = None):
        self.nx = num_qubit
        self.num_reads = num_reads

    def init(self):
        pass

TypeError: unsupported operand type(s) for |: 'type' and 'type'

In [1]:
import sys
sys.version

'3.8.13 (default, Mar 28 2022, 06:13:39) \n[Clang 12.0.0 ]'